# Análise em tempo real do consumo energético dos CPEs da CMMaia

Responde às DUAS perguntas centrais do sistema:
1. RETROSPETIVA — "Como foi ontem?"
2. PREDIÇÃO    — "O que esperar para amanhã?"


Calibração robusta:

- Threshold ADAPTATIVO por tipo de dia
    - Dia útil:     z > 2.0  (muitos dados, calibração fina)
    - Fim semana:   z > 2.5  (médio)
    - Feriado:      z > 3.0  (poucos dados, exige mais evidência)

- Fallback inteligente — se < 10 dias do mesmo tipo, usa todo o histórico mas marca o resultado como "baixa confiança"

- Z-score limitado a ±10 (acima disto é instabilidade estatística)

- Std mínimo proporcional ao habitual (10% do valor) em vez de fixo, para evitar inflar z-scores em CPEs muito constantes

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import sys
import zipfile
import warnings
from pathlib import Path
from datetime import date, timedelta

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import holidays

# Permite importar módulos internos de src/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from config.paths import (
    RESULTS_DIR,
    RESULTS_REALTIME_DIR,
    ALERTS_DIR,
    PREDICTIONS_DIR,
    ANALYSIS_DIR,
    RESULTS_CLUSTERING_DIR,
    ZIP_FALLBACK_PATH,
)

from data.baze_loader import carregar_dados_baze, CPES_CONFIG

warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (14, 5)
sns.set_theme(style="whitegrid")

# Paleta
COR_NORMAL = "#3498DB"
COR_DESVIO = "#E74C3C"
COR_MENOS = "#1ABC9C"
COR_PREDICAO = "#9B59B6"
COR_BAIXA_CONF = "#95A5A6"
COR_FUNDO = "#FAFBFC"

# Caminhos
base_dir = RESULTS_DIR
realtime_dir = RESULTS_REALTIME_DIR
alerts_dir = ALERTS_DIR
predictions_dir = PREDICTIONS_DIR
analysis_dir = ANALYSIS_DIR
clustering_dir = RESULTS_CLUSTERING_DIR

for d in [alerts_dir, predictions_dir, analysis_dir]:
    d.mkdir(parents=True, exist_ok=True)

# Parâmetros de deteção calibrados por tipo de dia
THRESHOLD_POR_TIPO = {
    "dia_util": 2.0,
    "fim_semana": 2.5,
    "feriado": 3.0,
}

MIN_HIST_IDEAL   = 10    # mín. de dias do mesmo tipo para confiança "alta"
MIN_HIST_ABSOLUTO = 3    # absoluto: abaixo disto descartar análise
Z_CAP            = 10.0  # limite máximo de |z| (evita números absurdos)
STD_MIN_FRAC     = 0.10  # std mínimo = 10% do habitual (não fixo)
STD_ABSOLUTO     = 0.3   # piso absoluto para evitar div/0
BASELINE_WINDOW_DAYS = 60  # janela móvel: usar só os últimos N dias (sazonalidade)

In [ ]:
# BLOCO 1 — Carregamento dos dados

print("=" * 70)
print("ANÁLISE EM TEMPO REAL — RETROSPETIVA + PREDIÇÃO")
print("=" * 70)
 
FONTE = "baze"   # "baze" para produção, "zip" para validação
 
print(f"\nFonte de dados: {FONTE.upper()}")
 
if FONTE == "baze":
    df_energia = carregar_dados_baze(CPES_CONFIG, anos=[2026], usar_cache=True)

else:
    zip_path = ZIP_FALLBACK_PATH

    with zipfile.ZipFile(zip_path) as z:
        with z.open(z.namelist()[0]) as f:
            df_raw = pd.read_csv(f)

    df_raw["tstamp"] = pd.to_datetime(df_raw["tstamp"])
    df_raw["data"] = df_raw["tstamp"].dt.date

    df_energia = (
        df_raw
        .groupby(["CPE", "data"])["PotActiva"]
        .mean()
        .reset_index()
    )

    df_energia["tstamp"] = pd.to_datetime(df_energia["data"])
 
df_energia = df_energia.sort_values(["CPE", "data"]).reset_index(drop=True)
 
print(f"  Registos: {len(df_energia):,}")
print(f"  CPEs:     {df_energia['CPE'].nunique()}")
print(f"  Período:  {df_energia['data'].min()} → {df_energia['data'].max()}")

In [ ]:
# BLOCO 2 — Classificação de tipo de dia + clusters

print("\n" + "─" * 70)
print("CLASSIFICAÇÃO DE TIPO DE DIA")
print("─" * 70)
 
anos = sorted({d.year for d in df_energia["data"].unique()})
feriados_pt = holidays.Portugal(years=anos)
 
def classificar_dia(d):
    if d in feriados_pt:
        return "feriado"
    if pd.Timestamp(d).dayofweek >= 5:
        return "fim_semana"
    return "dia_util"
 
df_energia["tipo_dia"] = df_energia["data"].map(classificar_dia)
 
# Clusters opcionais
try:
    clusters = pd.read_csv(clustering_dir / "clusters_cpe.csv", index_col=0)
    clusters = clusters[clusters["cluster"] != "outlier"]
    clusters["cluster"] = clusters["cluster"].astype(int)
    df_energia = df_energia.merge(
        clusters[["cluster"]], left_on="CPE", right_index=True, how="left"
    )
    print(f"  Clusters carregados: {df_energia['cluster'].nunique()}")
except Exception as e:
    df_energia["cluster"] = np.nan
    print(f"  Sem clusters: {e}")
 
print("\nDistribuição de tipos de dia:")
display(df_energia.drop_duplicates("data")["tipo_dia"].value_counts())
 
# Avisar se houver poucos dados de algum tipo
contagem = df_energia.drop_duplicates("data")["tipo_dia"].value_counts()
for tipo in ["dia_util", "fim_semana", "feriado"]:
    n = contagem.get(tipo, 0)
    if n < 30:
        print(f"  ⚠️  Apenas {n} {tipo.replace('_', ' ')}s no dataset — "
              f"baselines deste tipo serão menos fiáveis.")

In [ ]:
# BLOCO 3 — PARTE 1: RETROSPETIVA

print("\n" + "=" * 70)
print("PARTE 1 — RETROSPETIVA: 'Como foi ontem?'")
print("=" * 70)
 
data_ontem = max(df_energia["data"].unique())
tipo_ontem = classificar_dia(data_ontem)
threshold_ontem = THRESHOLD_POR_TIPO[tipo_ontem]
 
print(f"\nDia analisado: {data_ontem} ({tipo_ontem.replace('_', ' ')})")
print(f"Threshold para este tipo de dia: |z| > {threshold_ontem}")
 
def analisar_dia_cpe(cpe, data_alvo, df):
    """
    Compara o consumo de 'data_alvo' com o histórico do PRÓPRIO CPE
    para dias do mesmo tipo, com fallback inteligente e calibração robusta.

    Devolve: dict com resultado + nível de confiança ('alta' / 'baixa')
    """
    df_cpe = df[df["CPE"] == cpe].copy()
    if df_cpe.empty:
        return None

    consumo_real = df_cpe[df_cpe["data"] == data_alvo]["PotActiva"]
    if consumo_real.empty:
        return None
    consumo_real = float(consumo_real.iloc[0])

    tipo = classificar_dia(data_alvo)
    threshold = THRESHOLD_POR_TIPO[tipo]

    janela_inicio = data_alvo - timedelta(days=BASELINE_WINDOW_DAYS)
    df_hist_tipo = df_cpe[
        (df_cpe["data"] != data_alvo)
        & (df_cpe["tipo_dia"] == tipo)
        & (df_cpe["data"] >= janela_inicio)
    ]

    # Determinar nível de confiança
    if len(df_hist_tipo) >= MIN_HIST_IDEAL:
        df_hist = df_hist_tipo
        confianca = "alta"
        fonte_baseline = f"histórico {tipo}"
    elif len(df_hist_tipo) >= MIN_HIST_ABSOLUTO:
        df_hist = df_hist_tipo
        confianca = "baixa"
        fonte_baseline = f"histórico {tipo} (poucos dados)"
    else:
        df_hist = df_cpe[
            (df_cpe["data"] != data_alvo)
            & (df_cpe["data"] >= janela_inicio)
        ]
        if len(df_hist) < MIN_HIST_ABSOLUTO:
            return None
        confianca = "baixa"
        fonte_baseline = "fallback (todo histórico)"

    media = float(df_hist["PotActiva"].mean())
    std   = float(df_hist["PotActiva"].std())
    if pd.isna(std):
        std = 0.0

    # Std mínimo proporcional ao habitual (evita inflação em CPEs constantes)
    std = max(std, abs(media) * STD_MIN_FRAC, STD_ABSOLUTO)

    z = (consumo_real - media) / std
    z_capped = float(np.clip(z, -Z_CAP, Z_CAP))

    return {
        "CPE"           : cpe,
        "data"          : data_alvo,
        "tipo_dia"      : tipo,
        "consumo"       : round(consumo_real, 2),
        "habitual"      : round(media, 2),
        "std"           : round(std, 2),
        "z_score"       : round(z_capped, 2),
        "z_real"        : round(z, 2),
        "desvio_pct"    : round((consumo_real - media) / max(abs(media), 0.01) * 100, 1),
        "veredicto"     : "desvio" if abs(z_capped) > threshold else "normal",
        "n_dias"        : len(df_hist),
        "confianca"     : confianca,
        "fonte_baseline": fonte_baseline,
        "threshold"     : threshold,
    }
 
 
# Analisar todos os CPEs com dados para ontem
cpes_ontem = df_energia[df_energia["data"] == data_ontem]["CPE"].unique()
print(f"  CPEs com dados para ontem: {len(cpes_ontem)}")
 
resultados = []
for cpe in cpes_ontem:
    r = analisar_dia_cpe(cpe, data_ontem, df_energia)
    if r is not None:
        resultados.append(r)
 
df_retro = pd.DataFrame(resultados)
print(f"  CPEs analisados:           {len(df_retro)}")
 
# Resumo por confiança
n_alta_normal = ((df_retro["confianca"] == "alta") &
                 (df_retro["veredicto"] == "normal")).sum()
n_alta_desvio = ((df_retro["confianca"] == "alta") &
                 (df_retro["veredicto"] == "desvio")).sum()
n_baixa_normal = ((df_retro["confianca"] == "baixa") &
                  (df_retro["veredicto"] == "normal")).sum()
n_baixa_desvio = ((df_retro["confianca"] == "baixa") &
                  (df_retro["veredicto"] == "desvio")).sum()
 
print(f"\n  Confiança ALTA:   ✓ {n_alta_normal} normais   ✗ {n_alta_desvio} desvios")
print(f"  Confiança BAIXA:  ✓ {n_baixa_normal} normais   ✗ {n_baixa_desvio} desvios")
 
 
# ── Visualização A: distribuição dos z-scores ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5), facecolor=COR_FUNDO)
 
ax = axes[0]
ax.set_facecolor(COR_FUNDO)
 
# Separar por confiança
z_alta  = df_retro[df_retro["confianca"] == "alta"]["z_score"]
z_baixa = df_retro[df_retro["confianca"] == "baixa"]["z_score"]
 
bins = np.linspace(-Z_CAP, Z_CAP, 41)
if len(z_alta) > 0:
    ax.hist(z_alta, bins=bins, color=COR_NORMAL, alpha=0.75,
            edgecolor="white", label=f"Confiança alta ({len(z_alta)})")
if len(z_baixa) > 0:
    ax.hist(z_baixa, bins=bins, color=COR_BAIXA_CONF, alpha=0.65,
            edgecolor="white", label=f"Confiança baixa ({len(z_baixa)})")
 
ax.axvline( threshold_ontem, color=COR_DESVIO, linestyle="--",
            linewidth=1.5, label=f"|z| = {threshold_ontem}")
ax.axvline(-threshold_ontem, color=COR_DESVIO, linestyle="--", linewidth=1.5)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title(f"Distribuição dos z-scores ({data_ontem})", fontweight="bold")
ax.set_xlabel("Z-score (capped a ±10)")
ax.set_ylabel("Nº CPEs")
ax.legend()
 
# Top 10 maiores desvios
ax = axes[1]
ax.set_facecolor(COR_FUNDO)
top_desvios = df_retro.reindex(
    df_retro["z_score"].abs().sort_values(ascending=False).index
).head(10)
 
cores = []
for _, row in top_desvios.iterrows():
    if row["confianca"] == "baixa":
        cores.append(COR_BAIXA_CONF)
    elif row["z_score"] > 0:
        cores.append(COR_DESVIO)
    else:
        cores.append(COR_MENOS)
 
ax.barh(range(len(top_desvios)), top_desvios["z_score"],
        color=cores, alpha=0.85, edgecolor="white")
labels = [f"{cpe[-12:]}{' ⚠' if c == 'baixa' else ''}"
          for cpe, c in zip(top_desvios["CPE"], top_desvios["confianca"])]
ax.set_yticks(range(len(top_desvios)))
ax.set_yticklabels(labels, fontsize=9)
ax.axvline(0, color="black", linewidth=0.8)
ax.axvline( threshold_ontem, color=COR_DESVIO, linestyle="--", linewidth=1)
ax.axvline(-threshold_ontem, color=COR_DESVIO, linestyle="--", linewidth=1)
ax.set_title("Top 10 maiores desvios",
             fontweight="bold")
ax.set_xlabel("Z-score")
ax.invert_yaxis()
 
plt.tight_layout()
plt.savefig(analysis_dir / f"retrospetiva_{data_ontem}.png", dpi=130,
            bbox_inches="tight", facecolor=COR_FUNDO)
plt.show()
 
 
# ── Visualização B: detalhe dos desvios (separados por confiança) ──────────
df_desvios = df_retro[df_retro["veredicto"] == "desvio"].sort_values(
    "z_score", key=lambda x: x.abs(), ascending=False
)
 
if len(df_desvios) > 0:
    desvios_alta  = df_desvios[df_desvios["confianca"] == "alta"]
    desvios_baixa = df_desvios[df_desvios["confianca"] == "baixa"]
 
    if len(desvios_alta) > 0:
        print(f"\n  🔴 Desvios com ALTA confiança ({len(desvios_alta)}):")
        display(desvios_alta[["CPE", "tipo_dia", "consumo", "habitual",
                              "std", "z_score", "desvio_pct", "n_dias",
                              "fonte_baseline"]])
 
    if len(desvios_baixa) > 0:
        print(f"\n  ⚠️  Desvios com BAIXA confiança ({len(desvios_baixa)}):")
        print(f"     (verificar — baseado em poucos dados históricos)")
        display(desvios_baixa[["CPE", "tipo_dia", "consumo", "habitual",
                               "std", "z_score", "desvio_pct", "n_dias",
                               "fonte_baseline"]])
 
    # Para os top 4 com alta confiança (ou top 4 globais se não houver),
    # mostrar série temporal
    top_para_plot = desvios_alta.head(4) if len(desvios_alta) >= 4 \
                    else df_desvios.head(4)
 
    fig, axes = plt.subplots(2, 2, figsize=(18, 10), facecolor=COR_FUNDO)
    axes = axes.flatten()
 
    for ax, (_, row) in zip(axes, top_para_plot.iterrows()):
        ax.set_facecolor(COR_FUNDO)
        cpe = row["CPE"]
        df_cpe = df_energia[df_energia["CPE"] == cpe].sort_values("data")
 
        datas = pd.to_datetime(df_cpe["data"])
        ax.plot(datas, df_cpe["PotActiva"], color=COR_NORMAL,
                linewidth=1.5, alpha=0.7, marker="o", markersize=3)
 
        # Banda habitual
        ax.axhline(row["habitual"], color="#27AE60", linewidth=2,
                   linestyle="--", label=f"Habitual ({row['habitual']:.1f})")
        ax.axhspan(row["habitual"] - row["std"], row["habitual"] + row["std"],
                   alpha=0.15, color="#27AE60")
 
        cor = COR_DESVIO if row["z_score"] > 0 else COR_MENOS
        ax.plot(pd.Timestamp(data_ontem), row["consumo"],
                marker="*", markersize=20, color=cor, zorder=5,
                label=f"Ontem: {row['consumo']:.1f} (z={row['z_score']:+.1f})")
 
        conf_str = " ⚠ baixa conf" if row["confianca"] == "baixa" else ""
        ax.set_title(f"{cpe[-12:]}  |  desvio {row['desvio_pct']:+.0f}%{conf_str}",
                     fontweight="bold", color=cor)
        ax.set_ylabel("Consumo (kWh)")
        ax.legend(fontsize=9, loc="best")
        ax.grid(True, alpha=0.3)
        ax.xaxis.set_major_locator(mdates.AutoDateLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, fontsize=8)
 
    plt.suptitle(f"Top 4 desvios em {data_ontem}",
                 fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(analysis_dir / f"top_desvios_{data_ontem}.png", dpi=130,
                bbox_inches="tight", facecolor=COR_FUNDO)
    plt.show()
else:
    print("\n  ✓ Sem desvios significativos.")
 
# Exportar
df_retro.to_csv(analysis_dir / f"retrospetiva_{data_ontem}.csv", index=False)
print(f"\n  CSV: {analysis_dir / f'retrospetiva_{data_ontem}.csv'}")

In [ ]:
# === BLOCO 4 — PARTE 2: PREDIÇÃO (apenas LEITURA do CSV do script) ===========
 
print("\n\n" + "=" * 70)
print("PARTE 2 — PREDIÇÃO: 'O que esperar a seguir?'  (lido do script)")
print("=" * 70)
 
# A previsão é a do dia seguinte ao analisado na Parte 1 (data_ontem).
data_amanha = data_ontem + timedelta(days=1)
 
# Localizar o CSV de previsão gerado pelo script.
# 1.º tenta o dia esperado; se não existir, usa o mais recente disponível.
pred_path = predictions_dir / f"previsao_{data_amanha}.csv"
if not pred_path.exists():
    ficheiros_pred = sorted(predictions_dir.glob("previsao_*.csv"))
    if ficheiros_pred:
        pred_path   = ficheiros_pred[-1]
        data_amanha = pd.to_datetime(pred_path.stem.replace("previsao_", "")).date()
        print(f"\n (previsão de {data_ontem + timedelta(days=1)} ainda não existe; "
              f"a usar a mais recente: {data_amanha})")
    else:
        pred_path = None
 
if pred_path is None:
    df_pred = None
    print("\n  Sem ficheiros de previsão em predictions/.")
    print("  Corre primeiro:  python detetar_anomalias.py --modo baze")
else:
    df_pred = pd.read_csv(pred_path)
    tipo_amanha = classificar_dia(data_amanha)
 
    n_alta_p  = (df_pred["confianca"] == "alta").sum()
    n_baixa_p = (df_pred["confianca"] == "baixa").sum()
 
    print(f"\nDia previsto: {data_amanha} ({tipo_amanha.replace('_', ' ')})")
    print(f"  Origem: {pred_path}")
    print(f"  CPEs previstos: {len(df_pred)} "
          f"(alta confiança: {n_alta_p}, baixa: {n_baixa_p})")
    print(f"  Consumo total previsto: {df_pred['previsao'].sum():.1f} kWh "
          f"(±{df_pred['std'].sum():.1f})")
 
# ── Figuras (só se houver previsão) ──────────────────────────────────────────
if df_pred is not None:
    # Figura 1: distribuição das previsões + incerteza relativa
    fig, axes = plt.subplots(1, 2, figsize=(16, 5), facecolor=COR_FUNDO)
 
    ax = axes[0]
    ax.set_facecolor(COR_FUNDO)
    ax.hist(df_pred[df_pred["confianca"] == "alta"]["previsao"],
            bins=40, color=COR_PREDICAO, alpha=0.7, edgecolor="white",
            label=f"Alta conf ({n_alta_p})")
    if n_baixa_p > 0:
        ax.hist(df_pred[df_pred["confianca"] == "baixa"]["previsao"],
                bins=40, color=COR_BAIXA_CONF, alpha=0.6, edgecolor="white",
                label=f"Baixa conf ({n_baixa_p})")
    ax.set_title(f"Distribuição das previsões ({data_amanha})", fontweight="bold")
    ax.set_xlabel("Consumo previsto (kWh)")
    ax.set_ylabel("Nº CPEs")
    ax.set_yscale("log")
    ax.legend()
 
    ax = axes[1]
    ax.set_facecolor(COR_FUNDO)
    amplitude     = (df_pred["high_2sigma"] - df_pred["low_2sigma"]) / 2
    incerteza_rel = amplitude / df_pred["previsao"].clip(lower=0.01) * 100
    cores_scatter = [COR_BAIXA_CONF if c == "baixa" else COR_PREDICAO
                     for c in df_pred["confianca"]]
    ax.scatter(df_pred["previsao"], incerteza_rel,
               alpha=0.6, s=30, c=cores_scatter)
    ax.set_title("Incerteza relativa das previsões", fontweight="bold")
    ax.set_xlabel("Consumo previsto (kWh)")
    ax.set_ylabel("Incerteza (2σ / previsão, %)")
    ax.set_xscale("log")
    ax.set_yscale("log")
 
    plt.tight_layout()
    plt.savefig(analysis_dir / f"predicao_{data_amanha}.png", dpi=130,
                bbox_inches="tight", facecolor=COR_FUNDO)
    plt.show()
 
    # Figura 2: previsão vs histórico (amostra — top 4 consumos previstos)
    cpes_amostra = df_pred[df_pred["confianca"] == "alta"].sort_values(
        "previsao", ascending=False
    ).head(4)["CPE"].tolist()
    if len(cpes_amostra) == 0:
        cpes_amostra = df_pred.sort_values(
            "previsao", ascending=False
        ).head(4)["CPE"].tolist()
 
    fig, axes = plt.subplots(2, 2, figsize=(18, 10), facecolor=COR_FUNDO)
    axes = axes.flatten()
    for ax, cpe in zip(axes, cpes_amostra):
        ax.set_facecolor(COR_FUNDO)
        df_cpe = df_energia[df_energia["CPE"] == cpe].sort_values("data")
        pred   = df_pred[df_pred["CPE"] == cpe].iloc[0]
 
        datas = pd.to_datetime(df_cpe["data"])
        ax.plot(datas, df_cpe["PotActiva"], color=COR_NORMAL,
                linewidth=1.5, alpha=0.7, marker="o", markersize=3,
                label="Histórico")
 
        dt_amanha = pd.Timestamp(data_amanha)
        ax.errorbar(dt_amanha, pred["previsao"],
                    yerr=[[pred["previsao"] - pred["low_2sigma"]],
                          [pred["high_2sigma"] - pred["previsao"]]],
                    fmt="s", color=COR_PREDICAO, markersize=10,
                    capsize=8, linewidth=2,
                    label=f"Previsão: {pred['previsao']:.1f} ±{pred['std']:.1f}")
 
        conf_str = " ⚠" if pred["confianca"] == "baixa" else ""
        ax.set_title(f"{cpe[-12:]} | {tipo_amanha.replace('_', ' ')}{conf_str}",
                     fontweight="bold")
        ax.set_ylabel("Consumo (kWh)")
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.xaxis.set_major_locator(mdates.AutoDateLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, fontsize=8)
 
    plt.suptitle(f"Previsões para {data_amanha} (top 4 consumos)",
                 fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(analysis_dir / f"predicao_amostra_{data_amanha}.png", dpi=130,
                bbox_inches="tight", facecolor=COR_FUNDO)
    plt.show()

In [ ]:
# === BLOCO 5 — PARTE 3: VALIDAÇÃO (apenas LEITURA do CSV do script) ==========
 
print("\n\n" + "=" * 70)
print("PARTE 3 — VALIDAÇÃO: 'Quão boas foram as previsões?'  (lido do script)")
print("=" * 70)
 
# O script grava o histórico acumulado de qualidade aqui:
val_path = analysis_dir / "qualidade_previsoes.csv"
 
if not val_path.exists():
    print("\n  Ainda não há histórico de validação (qualidade_previsoes.csv).")
    print("  Isto preenche-se sozinho: cada dia que o script corre, valida as")
    print("  previsões cujo consumo real já chegou. Volta aqui depois de uns dias.")
else:
    df_val = pd.read_csv(val_path)
    df_val["data"] = pd.to_datetime(df_val["data"])
    df_val = df_val.sort_values("data").reset_index(drop=True)
 
    print(f"\n  Dias validados: {len(df_val)}   (origem: {val_path})")
    print("\n  Métricas de qualidade das previsões:")
    display(df_val)
 
    print(f"\n  Médias acumuladas:")
    print(f"    MAE  : {df_val['MAE'].mean():.2f} kWh")
    print(f"    RMSE : {df_val['RMSE'].mean():.2f} kWh")
    print(f"    em ±1σ: {df_val['pct_em_1sigma'].mean():.1f}%  (esperado ~68%)")
    print(f"    em ±2σ: {df_val['pct_em_2sigma'].mean():.1f}%  (esperado ~95%)")
 
    # Figuras só fazem sentido com 2+ pontos no tempo
    if len(df_val) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(16, 5), facecolor=COR_FUNDO)
 
        ax = axes[0]
        ax.set_facecolor(COR_FUNDO)
        ax.plot(df_val["data"], df_val["MAE"], marker="o",
                color=COR_DESVIO, linewidth=2, label="MAE")
        ax.plot(df_val["data"], df_val["RMSE"], marker="s",
                color=COR_PREDICAO, linewidth=2, label="RMSE")
        ax.set_title("Erro das previsões ao longo do tempo", fontweight="bold")
        ax.set_xlabel("Data prevista")
        ax.set_ylabel("Erro (kWh)")
        ax.legend()
        ax.grid(True, alpha=0.3)
 
        ax = axes[1]
        ax.set_facecolor(COR_FUNDO)
        ax.plot(df_val["data"], df_val["pct_em_1sigma"], marker="o",
                color="#27AE60", linewidth=2, label="dentro ±1σ")
        ax.plot(df_val["data"], df_val["pct_em_2sigma"], marker="s",
                color=COR_NORMAL, linewidth=2, label="dentro ±2σ")
        ax.axhline(68, color="#27AE60", linestyle=":", alpha=0.5,
                   label="esperado 68%")
        ax.axhline(95, color=COR_NORMAL, linestyle=":", alpha=0.5,
                   label="esperado 95%")
        ax.set_title("Cobertura dos intervalos de confiança", fontweight="bold")
        ax.set_xlabel("Data prevista")
        ax.set_ylabel("% CPEs dentro do intervalo")
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
 
        plt.tight_layout()
        # NOTA: gravamos só a FIGURA. O CSV qualidade_previsoes.csv é propriedade
        # do script — o notebook nunca o reescreve (evita dois donos do ficheiro).
        plt.savefig(analysis_dir / "qualidade_previsoes.png", dpi=130,
                    bbox_inches="tight", facecolor=COR_FUNDO)
        plt.show()
    else:
        print("\n  (Só há 1 dia validado — as curvas ao longo do tempo aparecem "
              "a partir do 2.º dia.)")